# Подготовка Windows Event Logs (Server A)

In [1]:
import os
import re
import csv
import json
import pickle
from pathlib import Path
import numpy as np
import pandas as pd

## Конфигурация эксперимента

In [2]:
WINDOW_SIZE = "30min"
MIN_SEQ_LEN = 5
MAX_CHUNK_LEN = 64
STRIDE = 32

TRAIN_SIZE = 0.70
VAL_SIZE = 0.15
TEST_SIZE = 0.15

FORCE_RECREATE_LOCKED_SPLIT = False

SAVE_DIR = "/content/drive/MyDrive/windows_logs_project"
VERSION = "v3_30min_detailed_tokens_recomputed_chunks"

## Подключение Google Drive

In [3]:
if IN_COLAB:
    drive.mount("/content/drive")
else:
    print("Not running in Google Colab. Make sure SAVE_DIR points to an existing local directory.")

os.makedirs(SAVE_DIR, exist_ok=True)

LOCKED_DATA_PATH = f"{SAVE_DIR}/windows_locked_split_30min_v3.pkl"
EXPORT_DIR = f"{SAVE_DIR}/export_30min_v3"
PROMPT_EXPORT_DIR = f"{SAVE_DIR}/annotation_prompts_30min_v3"

os.makedirs(EXPORT_DIR, exist_ok=True)
os.makedirs(PROMPT_EXPORT_DIR, exist_ok=True)

print("SAVE_DIR:", SAVE_DIR)
print("LOCKED_DATA_PATH:", LOCKED_DATA_PATH)
print("EXPORT_DIR:", EXPORT_DIR)
print("PROMPT_EXPORT_DIR:", PROMPT_EXPORT_DIR)

Mounted at /content/drive
SAVE_DIR: /content/drive/MyDrive/windows_logs_project
LOCKED_DATA_PATH: /content/drive/MyDrive/windows_logs_project/windows_locked_split_30min_v3.pkl
EXPORT_DIR: /content/drive/MyDrive/windows_logs_project/export_30min_v3
PROMPT_EXPORT_DIR: /content/drive/MyDrive/windows_logs_project/annotation_prompts_30min_v3


## Загрузка и первичная очистка датасета

In [4]:
WINDOWS_COLUMNS = [
    "Level",
    "Date and Time",
    "Source",
    "Event ID",
    "Task Category",
    "Message"
]


def load_windows_event_csv(file_path: str, log_name: str) -> pd.DataFrame:
    rows = []

    with open(file_path, "r", encoding="utf-8-sig", errors="replace", newline="") as f:
        reader = csv.reader(f)
        broken_header = next(reader, None)

        print(f"{log_name} | {Path(file_path).name} исходный header:", broken_header)

        for row in reader:
            if not row or all(not str(x).strip() for x in row):
                continue

            if len(row) == 6:
                rows.append(row)
            elif len(row) > 6:
                rows.append(row[:5] + [",".join(row[5:])])
            else:
                rows.append(row + [None] * (6 - len(row)))

    df = pd.DataFrame(rows, columns=WINDOWS_COLUMNS)
    df["LogName"] = log_name
    df["SourceFile"] = Path(file_path).name

    return df


def clean_windows_events(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    for col in ["Level", "Date and Time", "Source", "Event ID", "Task Category"]:
        df[col] = df[col].astype(str).str.strip()

    df["Message"] = df["Message"].fillna("").astype(str).str.strip()

    df["timestamp"] = pd.to_datetime(
        df["Date and Time"],
        errors="coerce",
        dayfirst=True
    )

    before = len(df)
    df = df.dropna(subset=["timestamp"]).copy()

    print(f"Удалено строк с некорректной датой: {before - len(df)}")

    return df


def load_many_windows_logs(files_by_logname: dict) -> pd.DataFrame:
    all_parts = []

    for log_name, paths in files_by_logname.items():
        for path in paths:
            if not Path(path).exists():
                print(f"Skip missing file: {path}")
                continue

            raw = load_windows_event_csv(path, log_name)
            clean = clean_windows_events(raw)
            all_parts.append(clean)

    if not all_parts:
        raise FileNotFoundError("No input log files were found. Check files_by_logname paths.")

    all_logs = pd.concat(all_parts, ignore_index=True)

    print("До удаления дубликатов:", len(all_logs))

    dedup_cols = [
        "LogName",
        "timestamp",
        "Level",
        "Source",
        "Event ID",
        "Task Category",
        "Message"
    ]

    duplicate_count = all_logs.duplicated(subset=dedup_cols).sum()
    print("Количество дубликатов:", duplicate_count)

    all_logs = all_logs.drop_duplicates(subset=dedup_cols, keep="first").copy()

    print("После удаления дубликатов:", len(all_logs))

    all_logs = all_logs.sort_values("timestamp").reset_index(drop=True)

    print("Диапазон дат:", all_logs["timestamp"].min(), "—", all_logs["timestamp"].max())

    return all_logs

In [6]:
files_by_logname = {
    "System": [
        "/content/system.csv",
        "/content/sys.csv"
    ],
    "Application": [
        "/content/application.csv",
        "/content/app.csv"
    ]
}

windows_clean = load_many_windows_logs(files_by_logname)

print("WINDOWS shape:", windows_clean.shape)

display(windows_clean.head(10))
display(windows_clean.tail(10))

System | system.csv исходный header: ['Level', 'Date and Time', 'Source', 'Event ID', 'Task Category']
Удалено строк с некорректной датой: 0
System | sys.csv исходный header: ['Level', 'Date and Time', 'Source', 'Event ID', 'Task Category']
Удалено строк с некорректной датой: 0
Application | application.csv исходный header: ['Level', 'Date and Time', 'Source', 'Event ID', 'Task Category']
Удалено строк с некорректной датой: 9
Application | app.csv исходный header: ['Level', 'Date and Time', 'Source', 'Event ID', 'Task Category']
Удалено строк с некорректной датой: 15
До удаления дубликатов: 182738
Количество дубликатов: 53648
После удаления дубликатов: 129090
Диапазон дат: 2026-01-25 04:37:59 — 2026-05-03 00:34:27
WINDOWS shape: (129090, 9)


,Level,Date and Time,Source,Event ID,Task Category,Message,LogName,SourceFile,timestamp
0,Information,25.01.2026 4:37:59,Service Control Manager,7036,None,The Network Setup Service service entered the ...,System,system.csv,2026-01-25 04:37:59
1,Information,25.01.2026 4:45:38,Service Control Manager,7036,None,The Network Setup Service service entered the ...,System,system.csv,2026-01-25 04:45:38
2,Information,25.01.2026 4:47:06,Service Control Manager,7036,None,The BITS service entered the running state.,System,system.csv,2026-01-25 04:47:06
3,Information,25.01.2026 4:47:06,Service Control Manager,7040,None,The start type of the BITS service was changed...,System,system.csv,2026-01-25 04:47:06
4,Information,25.01.2026 4:47:39,Service Control Manager,7036,None,The Network Setup Service service entered the ...,System,system.csv,2026-01-25 04:47:39
5,Information,25.01.2026 4:49:45,Service Control Manager,7036,None,The Network Setup Service service entered the ...,System,system.csv,2026-01-25 04:49:45
6,Information,25.01.2026 4:52:22,Service Control Manager,7040,None,The start type of the BITS service was changed...,System,system.csv,2026-01-25 04:52:22
7,Information,25.01.2026 4:52:22,Service Control Manager,7036,None,The BITS service entered the stopped state.,System,system.csv,2026-01-25 04:52:22
8,Information,25.01.2026 4:52:46,Service Control Manager,7036,None,The Network Setup Service service entered the ...,System,system.csv,2026-01-25 04:52:46
9,Information,25.01.2026 4:55:38,Service Control Manager,7036,None,The Network Setup Service service entered the ...,System,system.csv,2026-01-25 04:55:38


,Level,Date and Time,Source,Event ID,Task Category,Message,LogName,SourceFile,timestamp
129080,Information,03.05.2026 0:11:26,Service Control Manager,7036,None,The Network Setup Service service entered the ...,System,sys.csv,2026-05-03 00:11:26
129081,Information,03.05.2026 0:16:28,Service Control Manager,7036,None,The WMI Performance Adapter service entered th...,System,sys.csv,2026-05-03 00:16:28
129082,Information,03.05.2026 0:19:27,Service Control Manager,7036,None,The WMI Performance Adapter service entered th...,System,sys.csv,2026-05-03 00:19:27
129083,Information,03.05.2026 0:21:28,Service Control Manager,7036,None,The WMI Performance Adapter service entered th...,System,sys.csv,2026-05-03 00:21:28
129084,Information,03.05.2026 0:24:27,Service Control Manager,7036,None,The WMI Performance Adapter service entered th...,System,sys.csv,2026-05-03 00:24:27
129085,Information,03.05.2026 0:26:28,Service Control Manager,7036,None,The WMI Performance Adapter service entered th...,System,sys.csv,2026-05-03 00:26:28
129086,Information,03.05.2026 0:29:27,Service Control Manager,7036,None,The WMI Performance Adapter service entered th...,System,sys.csv,2026-05-03 00:29:27
129087,Information,03.05.2026 0:31:28,Service Control Manager,7036,None,The WMI Performance Adapter service entered th...,System,sys.csv,2026-05-03 00:31:28
129088,Information,03.05.2026 0:32:44,LsaSrv,45058,Logon Cache,A logon cache entry for user MMorozov@MOS.RENI...,System,sys.csv,2026-05-03 00:32:44
129089,Information,03.05.2026 0:34:27,Service Control Manager,7036,None,The WMI Performance Adapter service entered th...,System,sys.csv,2026-05-03 00:34:27


## Нормализация сообщений
(убираем значения, которые меняются, но не несут важной информации)

In [7]:
def normalize_message(text: str) -> str:
    text = str(text).lower()

    text = re.sub(
        r"\b[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}\b",
        "<guid>",
        text
    )

    text = re.sub(
        r"\b(?:\d{1,3}\.){3}\d{1,3}\b",
        "<ip>",
        text
    )
    text = re.sub(
        r"[a-z]:\\(?:[^\\/:*?\"<>|\r\n]+\\)*[^\\/:*?\"<>|\r\n]*",
        "<path>",
        text
    )

    text = re.sub(
        r"\b\d{1,2}[./-]\d{1,2}[./-]\d{2,4}\b",
        "<date>",
        text
    )

    text = re.sub(
        r"\b\d{1,2}:\d{2}(:\d{2})?\b",
        "<time>",
        text
    )

    text = re.sub(
        r"\b0x[0-9a-f]+\b",
        "<hex>",
        text
    )


    text = re.sub(
        r"\b\d+\b",
        "<num>",
        text
    )


    text = re.sub(r"\s+", " ", text).strip()

    return text

In [8]:
windows_clean["NormalizedMessage"] = windows_clean["Message"].apply(normalize_message)

display(windows_clean[[
    "timestamp",
    "LogName",
    "Level",
    "Source",
    "Event ID",
    "Task Category",
    "Message",
    "NormalizedMessage"
]].head(10))

,timestamp,LogName,Level,Source,Event ID,Task Category,Message,NormalizedMessage
0,2026-01-25 04:37:59,System,Information,Service Control Manager,7036,None,The Network Setup Service service entered the ...,the network setup service service entered the ...
1,2026-01-25 04:45:38,System,Information,Service Control Manager,7036,None,The Network Setup Service service entered the ...,the network setup service service entered the ...
2,2026-01-25 04:47:06,System,Information,Service Control Manager,7036,None,The BITS service entered the running state.,the bits service entered the running state.
3,2026-01-25 04:47:06,System,Information,Service Control Manager,7040,None,The start type of the BITS service was changed...,the start type of the bits service was changed...
4,2026-01-25 04:47:39,System,Information,Service Control Manager,7036,None,The Network Setup Service service entered the ...,the network setup service service entered the ...
5,2026-01-25 04:49:45,System,Information,Service Control Manager,7036,None,The Network Setup Service service entered the ...,the network setup service service entered the ...
6,2026-01-25 04:52:22,System,Information,Service Control Manager,7040,None,The start type of the BITS service was changed...,the start type of the bits service was changed...
7,2026-01-25 04:52:22,System,Information,Service Control Manager,7036,None,The BITS service entered the stopped state.,the bits service entered the stopped state.
8,2026-01-25 04:52:46,System,Information,Service Control Manager,7036,None,The Network Setup Service service entered the ...,the network setup service service entered the ...
9,2026-01-25 04:55:38,System,Information,Service Control Manager,7036,None,The Network Setup Service service entered the ...,the network setup service service entered the ...


## Словарь событий

In [9]:
windows_clean["EventToken"] = windows_clean.apply(
    lambda row: (
        f"{row['LogName']} | "
        f"{row['Source']} | "
        f"{row['Event ID']} | "
        f"{row['Level']} | "
        f"{row['NormalizedMessage']}"
    ),
    axis=1
)

windows_clean["EventTemplateForReview"] = windows_clean["EventToken"]

print("Уникальных EventToken:", windows_clean["EventToken"].nunique())

display(windows_clean[[
    "timestamp",
    "LogName",
    "Level",
    "Source",
    "Event ID",
    "EventToken"
]].head(10))

Уникальных EventToken: 3026


,timestamp,LogName,Level,Source,Event ID,EventToken
0,2026-01-25 04:37:59,System,Information,Service Control Manager,7036,System | Service Control Manager | 7036 | Info...
1,2026-01-25 04:45:38,System,Information,Service Control Manager,7036,System | Service Control Manager | 7036 | Info...
2,2026-01-25 04:47:06,System,Information,Service Control Manager,7036,System | Service Control Manager | 7036 | Info...
3,2026-01-25 04:47:06,System,Information,Service Control Manager,7040,System | Service Control Manager | 7040 | Info...
4,2026-01-25 04:47:39,System,Information,Service Control Manager,7036,System | Service Control Manager | 7036 | Info...
5,2026-01-25 04:49:45,System,Information,Service Control Manager,7036,System | Service Control Manager | 7036 | Info...
6,2026-01-25 04:52:22,System,Information,Service Control Manager,7040,System | Service Control Manager | 7040 | Info...
7,2026-01-25 04:52:22,System,Information,Service Control Manager,7036,System | Service Control Manager | 7036 | Info...
8,2026-01-25 04:52:46,System,Information,Service Control Manager,7036,System | Service Control Manager | 7036 | Info...
9,2026-01-25 04:55:38,System,Information,Service Control Manager,7036,System | Service Control Manager | 7036 | Info...


In [10]:
unique_tokens = (
    windows_clean["EventToken"]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

template_to_id = {
    token: f"W{i + 1}"
    for i, token in enumerate(unique_tokens)
}

id_to_template = {
    v: k
    for k, v in template_to_id.items()
}

windows_clean["EventTemplateID"] = windows_clean["EventToken"].map(template_to_id)

print("Количество шаблонов Windows:", len(template_to_id))

display(windows_clean[[
    "timestamp",
    "LogName",
    "Level",
    "Source",
    "Event ID",
    "EventTemplateID",
    "EventToken"
]].head(10))

Количество шаблонов Windows: 3026


,timestamp,LogName,Level,Source,Event ID,EventTemplateID,EventToken
0,2026-01-25 04:37:59,System,Information,Service Control Manager,7036,W2821,System | Service Control Manager | 7036 | Info...
1,2026-01-25 04:45:38,System,Information,Service Control Manager,7036,W2820,System | Service Control Manager | 7036 | Info...
2,2026-01-25 04:47:06,System,Information,Service Control Manager,7036,W2690,System | Service Control Manager | 7036 | Info...
3,2026-01-25 04:47:06,System,Information,Service Control Manager,7040,W2985,System | Service Control Manager | 7040 | Info...
4,2026-01-25 04:47:39,System,Information,Service Control Manager,7036,W2821,System | Service Control Manager | 7036 | Info...
5,2026-01-25 04:49:45,System,Information,Service Control Manager,7036,W2820,System | Service Control Manager | 7036 | Info...
6,2026-01-25 04:52:22,System,Information,Service Control Manager,7040,W2984,System | Service Control Manager | 7040 | Info...
7,2026-01-25 04:52:22,System,Information,Service Control Manager,7036,W2691,System | Service Control Manager | 7036 | Info...
8,2026-01-25 04:52:46,System,Information,Service Control Manager,7036,W2821,System | Service Control Manager | 7036 | Info...
9,2026-01-25 04:55:38,System,Information,Service Control Manager,7036,W2820,System | Service Control Manager | 7036 | Info...


##Исключаем явные аномалии


In [11]:
HARD_CRITICAL_EVENT_IDS = {
    "41",
    "6008",
    "7031",
    "7034",
    "17832",
    "17883",
    "1102",
    "4625",
    "4740",
    "9002",
    "4014",
    "18456"
}

SOFT_SUSPICIOUS_EVENT_IDS = {
    "1000",
    "1001",
    "10010",
    "10036",
    "1801",
    "1202",
    "1065",
    "8193"
}

## Формирование последовательностей по временным окнам


In [12]:
def build_sequences_by_time_window(
    df: pd.DataFrame,
    window: str = "30min",
    min_seq_len: int = 5
) -> pd.DataFrame:
    df = df.copy()
    df = df.sort_values("timestamp").reset_index(drop=True)

    df["window_start"] = df["timestamp"].dt.floor(window)

    sequences = []

    for window_start, group in df.groupby("window_start"):
        group = group.sort_values("timestamp")

        features = group["EventTemplateID"].tolist()
        timestamps = group["timestamp"].tolist()

        if len(features) < min_seq_len:
            continue

        time_intervals = [0.0]

        for i in range(1, len(timestamps)):
            delta = (timestamps[i] - timestamps[i - 1]).total_seconds()
            time_intervals.append(float(delta))

        latency = float((timestamps[-1] - timestamps[0]).total_seconds())

        sequences.append({
            "WindowSize": window,
            "WindowStart": window_start,
            "WindowEnd": window_start + pd.Timedelta(window),

            "Features": features,
            "TimeInterval": time_intervals,
            "Latency": latency,
            "SeqLen": len(features),

            "LogNames": sorted(group["LogName"].astype(str).unique().tolist()),
            "Sources": sorted(group["Source"].astype(str).unique().tolist()),
            "Levels": sorted(group["Level"].astype(str).unique().tolist()),
            "EventIDs": sorted(group["Event ID"].astype(str).unique().tolist()),

            "EventLogNames": group["LogName"].astype(str).tolist(),
            "EventSources": group["Source"].astype(str).tolist(),
            "EventLevels": group["Level"].astype(str).tolist(),
            "EventIDsPerEvent": group["Event ID"].astype(str).tolist(),
            "EventTokens": group["EventToken"].astype(str).tolist(),

            "RawMessages": group["Message"].astype(str).tolist(),
            "ReviewTemplates": group["EventTemplateForReview"].astype(str).tolist(),

            "Label": None
        })

    return pd.DataFrame(sequences)

In [13]:
def recompute_weak_label_for_row(row, q95_len=None):
    levels = [str(x).lower() for x in row["Levels"]]
    event_ids = set(str(x) for x in row["EventIDs"])
    sources = [str(x).lower() for x in row["Sources"]]

    hard_reasons = []
    soft_reasons = []

    if any("critical" in x for x in levels):
        hard_reasons.append("critical_level")

    matched_hard_ids = event_ids.intersection(HARD_CRITICAL_EVENT_IDS)
    if matched_hard_ids:
        hard_reasons.append(
            "hard_critical_event_id:" + ",".join(sorted(matched_hard_ids))
        )

    if any("mssqlserver" in s for s in sources) and any("error" in x for x in levels):
        hard_reasons.append("mssql_error")

    if any("error" in x for x in levels):
        soft_reasons.append("error_level")

    matched_soft_ids = event_ids.intersection(SOFT_SUSPICIOUS_EVENT_IDS)
    if matched_soft_ids:
        soft_reasons.append(
            "soft_suspicious_event_id:" + ",".join(sorted(matched_soft_ids))
        )

    if q95_len is not None and row["SeqLen"] > q95_len:
        soft_reasons.append("high_event_volume")

    weak_label = int(len(hard_reasons) > 0 or len(soft_reasons) > 0)
    hard_weak_label = int(len(hard_reasons) > 0)

    row["WeakLabel"] = weak_label
    row["HardWeakLabel"] = hard_weak_label
    row["HardSuspiciousReason"] = "; ".join(hard_reasons)
    row["SoftSuspiciousReason"] = "; ".join(soft_reasons)
    row["SuspiciousReason"] = "; ".join(hard_reasons + soft_reasons)
    row["NeedsReview"] = weak_label == 1

    return row


def add_weak_labels(windows_seq: pd.DataFrame) -> pd.DataFrame:
    windows_seq = windows_seq.copy()

    q95_len = windows_seq["SeqLen"].quantile(0.95)

    rows = []

    for _, row in windows_seq.iterrows():
        rows.append(recompute_weak_label_for_row(row.copy(), q95_len=q95_len))

    return pd.DataFrame(rows).reset_index(drop=True)

##Сравнение

In [14]:
windows_seq_15 = build_sequences_by_time_window(
    windows_clean,
    window="15min",
    min_seq_len=MIN_SEQ_LEN
)

windows_seq_30 = build_sequences_by_time_window(
    windows_clean,
    window="30min",
    min_seq_len=MIN_SEQ_LEN
)

windows_seq_15 = add_weak_labels(windows_seq_15)
windows_seq_30 = add_weak_labels(windows_seq_30)

print("15 минут:", windows_seq_15.shape)
print("30 минут:", windows_seq_30.shape)

15 минут: (8639, 25)
30 минут: (4696, 25)


In [15]:
def summarize_windows_sequences(name, seq_df):
    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    print("Количество окон:", len(seq_df))
    print("Диапазон времени:", seq_df["WindowStart"].min(), "—", seq_df["WindowEnd"].max())

    print("\nДлины последовательностей:")
    display(seq_df["SeqLen"].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

    print("\nWeakLabel distribution:")
    print(seq_df["WeakLabel"].value_counts(dropna=False))
    print(seq_df["WeakLabel"].value_counts(normalize=True, dropna=False))

    print("\nТоп-10 самых длинных окон:")
    display(
        seq_df.sort_values("SeqLen", ascending=False).head(10)[[
            "WindowStart",
            "WindowEnd",
            "SeqLen",
            "Levels",
            "EventIDs",
            "WeakLabel",
            "SuspiciousReason"
        ]]
    )


summarize_windows_sequences("Окна 15 минут", windows_seq_15)
summarize_windows_sequences("Окна 30 минут", windows_seq_30)


Окна 15 минут
Количество окон: 8639
Диапазон времени: 2026-01-25 04:45:00 — 2026-05-03 00:30:00

Длины последовательностей:


,SeqLen
count,8639.000000
mean,14.635027
std,23.524064
min,5.000000
50%,10.000000
75%,17.000000
90%,24.000000
95%,30.000000
99%,61.000000
max,771.000000



WeakLabel distribution:
WeakLabel
0    4363
1    4276
Name: count, dtype: int64
WeakLabel
0    0.505035
1    0.494965
Name: proportion, dtype: float64

Топ-10 самых длинных окон:


,WindowStart,WindowEnd,SeqLen,Levels,EventIDs,WeakLabel,SuspiciousReason
4467,2026-03-17 22:30:00,2026-03-17 22:45:00,771,"[Error, Information, Warning]","[1, 1003, 101, 1014, 10148, 10154, 1035, 1066,...",1,"hard_critical_event_id:18456,7031; mssql_error..."
5760,2026-03-31 23:00:00,2026-03-31 23:15:00,733,"[Error, Information, Warning]","[1, 1001, 10010, 1003, 101, 1014, 10148, 10149...",1,"hard_critical_event_id:18456,7031; mssql_error..."
7930,2026-04-24 23:00:00,2026-04-24 23:15:00,730,"[Error, Information, Warning]","[1, 10002, 1003, 101, 1014, 10148, 10149, 1015...",1,"hard_critical_event_id:17883,4625,7031; mssql_..."
6947,2026-04-14 11:15:00,2026-04-14 11:30:00,698,"[Error, Information, Warning]","[1, 10010, 1003, 101, 1014, 10148, 10149, 1015...",1,"hard_critical_event_id:17832,17883,18456,7031;..."
8092,2026-04-26 20:00:00,2026-04-26 20:15:00,456,"[Error, Information]","[17053, 18100, 18270, 3014, 3619, 5901, 7036, ...",1,hard_critical_event_id:9002; mssql_error; erro...
8089,2026-04-26 19:15:00,2026-04-26 19:30:00,450,"[Error, Information]","[3619, 7036, 7040, 898, 9002]",1,hard_critical_event_id:9002; mssql_error; erro...
8090,2026-04-26 19:30:00,2026-04-26 19:45:00,446,"[Error, Information]","[18270, 3014, 3619, 7036, 7040, 9002]",1,hard_critical_event_id:9002; mssql_error; erro...
8091,2026-04-26 19:45:00,2026-04-26 20:00:00,438,"[Error, Information]","[3619, 7036, 9002]",1,hard_critical_event_id:9002; mssql_error; erro...
8329,2026-04-29 14:30:00,2026-04-29 14:45:00,431,"[Error, Information, Warning]","[3619, 5612, 7036, 9002]",1,hard_critical_event_id:9002; mssql_error; erro...
2323,2026-02-21 17:45:00,2026-02-21 18:00:00,414,"[Error, Information, Warning]","[1, 1000, 1001, 1003, 1035, 1066, 10981, 1202,...",1,mssql_error; error_level; soft_suspicious_even...



Окна 30 минут
Количество окон: 4696
Диапазон времени: 2026-01-25 04:30:00 — 2026-05-03 00:30:00

Длины последовательностей:


,SeqLen
count,4696.000000
mean,27.488714
std,37.860833
min,6.000000
50%,23.000000
75%,31.000000
90%,40.000000
95%,48.000000
99%,120.150000
max,1029.000000



WeakLabel distribution:
WeakLabel
1    2861
0    1835
Name: count, dtype: int64
WeakLabel
1    0.609242
0    0.390758
Name: proportion, dtype: float64

Топ-10 самых длинных окон:


,WindowStart,WindowEnd,SeqLen,Levels,EventIDs,WeakLabel,SuspiciousReason
2484,2026-03-17 22:30:00,2026-03-17 23:00:00,1029,"[Error, Information, Warning]","[1, 1003, 101, 1014, 10148, 10154, 1035, 1066,...",1,"hard_critical_event_id:18456,7031; mssql_error..."
4398,2026-04-26 19:30:00,2026-04-26 20:00:00,884,"[Error, Information]","[18270, 3014, 3619, 7036, 7040, 9002]",1,hard_critical_event_id:9002; mssql_error; erro...
3805,2026-04-14 11:00:00,2026-04-14 11:30:00,805,"[Error, Information, Warning]","[1, 10010, 1003, 101, 1014, 10148, 10149, 1015...",1,"hard_critical_event_id:17832,17883,18456,4014,..."
3157,2026-03-31 23:00:00,2026-03-31 23:30:00,764,"[Error, Information, Warning]","[1, 1001, 10010, 1003, 101, 1014, 10148, 10149...",1,"hard_critical_event_id:18456,7031; mssql_error..."
4309,2026-04-24 23:00:00,2026-04-24 23:30:00,759,"[Error, Information, Warning]","[1, 10002, 1003, 101, 1014, 10148, 10149, 1015...",1,"hard_critical_event_id:17883,4625,7031; mssql_..."
1322,2026-02-21 17:30:00,2026-02-21 18:00:00,659,"[Error, Information, Warning]","[1, 1000, 1001, 1003, 10148, 10154, 1035, 1066...",1,"hard_critical_event_id:4625,7031; mssql_error;..."
4399,2026-04-26 20:00:00,2026-04-26 20:30:00,626,"[Error, Information]","[17053, 18100, 18270, 3014, 3619, 5901, 7036, ...",1,hard_critical_event_id:9002; mssql_error; erro...
2485,2026-03-17 23:00:00,2026-03-17 23:30:00,579,[Information],"[18456, 3450, 7036, 7040]",1,hard_critical_event_id:18456; high_event_volume
4532,2026-04-29 14:30:00,2026-04-29 15:00:00,534,"[Error, Information, Warning]","[3619, 45058, 5084, 5612, 7036, 9002]",1,hard_critical_event_id:9002; mssql_error; erro...
4397,2026-04-26 19:00:00,2026-04-26 19:30:00,520,"[Error, Information, Warning]","[1202, 1502, 158, 17053, 18100, 3619, 7036, 70...",1,hard_critical_event_id:9002; mssql_error; erro...


In [16]:
comparison_rows = []

for name, seq_df in [
    ("15min", windows_seq_15),
    ("30min", windows_seq_30)
]:
    comparison_rows.append({
        "window": name,
        "num_windows": len(seq_df),
        "mean_len": seq_df["SeqLen"].mean(),
        "median_len": seq_df["SeqLen"].median(),
        "p90_len": seq_df["SeqLen"].quantile(0.90),
        "p95_len": seq_df["SeqLen"].quantile(0.95),
        "p99_len": seq_df["SeqLen"].quantile(0.99),
        "max_len": seq_df["SeqLen"].max(),
        "weak_anomaly_count": int(seq_df["WeakLabel"].sum()),
        "weak_anomaly_ratio": seq_df["WeakLabel"].mean()
    })

window_comparison = pd.DataFrame(comparison_rows)

display(window_comparison)

,window,num_windows,mean_len,median_len,p90_len,p95_len,p99_len,max_len,weak_anomaly_count,weak_anomaly_ratio
0,15min,8639,14.635027,10.0,24.0,30.0,61.00,771,4276,0.494965
1,30min,4696,27.488714,23.0,40.0,48.0,120.15,1029,2861,0.609242


In [17]:
windows_seq = windows_seq_30.copy()

print("Используем окно:", WINDOW_SIZE)
print("windows_seq:", windows_seq.shape)

Используем окно: 30min
windows_seq: (4696, 25)


##Отбор нормальных окон для train

In [18]:
def has_level(row, level_name: str) -> bool:
    return any(level_name.lower() in str(x).lower() for x in row["Levels"])


def is_safe_normal(row, q95_len_train) -> bool:
    levels = [str(x).lower() for x in row["Levels"]]
    sources = [str(x).lower() for x in row["Sources"]]
    event_ids = set(str(x) for x in row["EventIDs"])

    has_critical_level = any("critical" in x for x in levels)

    has_hard_critical_event_id = (
        len(event_ids.intersection(HARD_CRITICAL_EVENT_IDS)) > 0
    )

    high_volume = row["SeqLen"] > q95_len_train

    has_error_level = any("error" in x for x in levels)

    serious_error_source = (
        any("mssqlserver" in s for s in sources)
        or any("service control manager" in s for s in sources)
        or any("kernel-power" in s for s in sources)
        or any("eventlog" in s for s in sources)
        or any("sqlserveragent" in s for s in sources)
    )

    has_serious_error = has_error_level and serious_error_source

    return (
        not has_critical_level
        and not has_hard_critical_event_id
        and not high_volume
        and not has_serious_error
    )

In [19]:
def split_long_sequence_row(row, max_len=64, stride=32, q95_len_for_weak=None):
    n = len(row["Features"])

    per_event_cols = [
        "Features",
        "TimeInterval",
        "RawMessages",
        "ReviewTemplates",
        "EventLogNames",
        "EventSources",
        "EventLevels",
        "EventIDsPerEvent",
        "EventTokens"
    ]

    for col in per_event_cols:
        if col not in row:
            raise ValueError(f"В row отсутствует колонка {col}. Проверь build_sequences_by_time_window.")

    if n <= max_len:
        new_row = row.copy()

        new_row["ChunkID"] = 0
        new_row["OriginalSeqLen"] = n
        new_row["ChunkStartPos"] = 0
        new_row["ChunkEndPos"] = n
        new_row["WasChunked"] = False

        new_row["LogNames"] = sorted(set(map(str, new_row["EventLogNames"])))
        new_row["Sources"] = sorted(set(map(str, new_row["EventSources"])))
        new_row["Levels"] = sorted(set(map(str, new_row["EventLevels"])))
        new_row["EventIDs"] = sorted(set(map(str, new_row["EventIDsPerEvent"])))

        new_row = recompute_weak_label_for_row(
            new_row,
            q95_len=q95_len_for_weak
        )

        return [new_row]

    chunks = []

    starts = list(range(0, n - max_len + 1, stride))
    last_start = n - max_len

    if starts[-1] != last_start:
        starts.append(last_start)

    for chunk_id, start in enumerate(starts):
        end = start + max_len

        new_row = row.copy()

        for col in per_event_cols:
            new_row[col] = row[col][start:end]

        new_row["SeqLen"] = len(new_row["Features"])
        new_row["Latency"] = float(sum(new_row["TimeInterval"]))

        new_row["LogNames"] = sorted(set(map(str, new_row["EventLogNames"])))
        new_row["Sources"] = sorted(set(map(str, new_row["EventSources"])))
        new_row["Levels"] = sorted(set(map(str, new_row["EventLevels"])))
        new_row["EventIDs"] = sorted(set(map(str, new_row["EventIDsPerEvent"])))

        new_row["ChunkID"] = chunk_id
        new_row["OriginalSeqLen"] = n
        new_row["ChunkStartPos"] = start
        new_row["ChunkEndPos"] = end
        new_row["WasChunked"] = True

        new_row = recompute_weak_label_for_row(
            new_row,
            q95_len=q95_len_for_weak
        )

        chunks.append(new_row)

    return chunks


def split_long_sequences(df, max_len=64, stride=32, q95_len_for_weak=None):
    all_rows = []

    for _, row in df.iterrows():
        all_rows.extend(
            split_long_sequence_row(
                row,
                max_len=max_len,
                stride=stride,
                q95_len_for_weak=q95_len_for_weak
            )
        )

    return pd.DataFrame(all_rows).reset_index(drop=True)

## Split

In [20]:
if FORCE_RECREATE_LOCKED_SPLIT and os.path.exists(LOCKED_DATA_PATH):
    os.remove(LOCKED_DATA_PATH)
    print("Удалён старый locked split:", LOCKED_DATA_PATH)


if os.path.exists(LOCKED_DATA_PATH):
    print("Найден locked split. Загружаю существующее разбиение...")

    with open(LOCKED_DATA_PATH, "rb") as f:
        locked_data = pickle.load(f)

    train_seq_raw = locked_data["train_seq_raw"]
    val_seq_raw = locked_data["val_seq_raw"]
    test_seq_raw = locked_data["test_seq_raw"]

    train_normal_raw = locked_data["train_normal_raw"]

    train_normal = locked_data["train_normal"]
    val_seq = locked_data["val_seq"]
    test_seq = locked_data["test_seq"]

    annotation_df = locked_data["annotation_df"]

    template_to_id = locked_data["template_to_id"]
    id_to_template = locked_data["id_to_template"]

    print("Loaded locked data:")
    print("Train raw:", train_seq_raw.shape)
    print("Train normal raw:", train_normal_raw.shape)
    print("Train normal chunked:", train_normal.shape)
    print("Val chunked:", val_seq.shape)
    print("Test chunked:", test_seq.shape)
    print("Annotation:", annotation_df.shape)

else:
    print("Locked split не найден. Создаю новое фиксированное разбиение...")

    windows_seq_sorted = windows_seq.sort_values("WindowStart").reset_index(drop=True)

    n = len(windows_seq_sorted)

    train_end = int(n * TRAIN_SIZE)
    val_end = int(n * (TRAIN_SIZE + VAL_SIZE))

    train_seq_raw = windows_seq_sorted.iloc[:train_end].copy()
    val_seq_raw = windows_seq_sorted.iloc[train_end:val_end].copy()
    test_seq_raw = windows_seq_sorted.iloc[val_end:].copy()

    print("Всего окон:", n)
    print("Train raw:", train_seq_raw.shape)
    print("Val raw:", val_seq_raw.shape)
    print("Test raw:", test_seq_raw.shape)

    print("\nДиапазоны времени:")
    print("Train:", train_seq_raw["WindowStart"].min(), "—", train_seq_raw["WindowEnd"].max())
    print("Val:", val_seq_raw["WindowStart"].min(), "—", val_seq_raw["WindowEnd"].max())
    print("Test:", test_seq_raw["WindowStart"].min(), "—", test_seq_raw["WindowEnd"].max())

    assert train_seq_raw["WindowEnd"].max() <= val_seq_raw["WindowStart"].min()
    assert val_seq_raw["WindowEnd"].max() <= test_seq_raw["WindowStart"].min()

    q95_len_train = train_seq_raw["SeqLen"].quantile(0.95)

    print("\n95-й перцентиль длины train-окон:", q95_len_train)

    train_seq_raw["SafeNormal"] = train_seq_raw.apply(
        lambda row: is_safe_normal(row, q95_len_train),
        axis=1
    )

    train_normal_raw = train_seq_raw[train_seq_raw["SafeNormal"]].copy()

    print("\nTrain raw всего:", len(train_seq_raw))
    print("Train safe-normal:", len(train_normal_raw))
    print(train_seq_raw["SafeNormal"].value_counts(dropna=False))
    print(train_seq_raw["SafeNormal"].value_counts(normalize=True, dropna=False))

    assert train_normal_raw["SafeNormal"].all()

    train_normal = split_long_sequences(
        train_normal_raw,
        max_len=MAX_CHUNK_LEN,
        stride=STRIDE,
        q95_len_for_weak=q95_len_train
    )

    val_seq = split_long_sequences(
        val_seq_raw,
        max_len=MAX_CHUNK_LEN,
        stride=STRIDE,
        q95_len_for_weak=q95_len_train
    )

    test_seq = split_long_sequences(
        test_seq_raw,
        max_len=MAX_CHUNK_LEN,
        stride=STRIDE,
        q95_len_for_weak=q95_len_train
    )

    print("\nПосле chunking:")
    print("Train normal raw:", train_normal_raw.shape, "->", train_normal.shape)
    print("Val raw:", val_seq_raw.shape, "->", val_seq.shape)
    print("Test raw:", test_seq_raw.shape, "->", test_seq.shape)

    print("\nМаксимальные длины после chunking:")
    print("Train:", train_normal["SeqLen"].max())
    print("Val:", val_seq["SeqLen"].max())
    print("Test:", test_seq["SeqLen"].max())

    annotation_df = pd.concat([
        val_seq.assign(Split="val"),
        test_seq.assign(Split="test")
    ]).reset_index(drop=True)

    annotation_df["AnnotationID"] = range(len(annotation_df))

    annotation_df["StableID"] = (
        annotation_df["Split"].astype(str) + "_" +
        annotation_df["WindowStart"].astype(str) + "_" +
        annotation_df["WindowEnd"].astype(str) + "_" +
        annotation_df["ChunkID"].astype(str) + "_" +
        annotation_df["ChunkStartPos"].astype(str) + "_" +
        annotation_df["ChunkEndPos"].astype(str)
    )

    locked_data = {
        "params": {
            "VERSION": VERSION,
            "WINDOW_SIZE": WINDOW_SIZE,
            "MAX_CHUNK_LEN": MAX_CHUNK_LEN,
            "STRIDE": STRIDE,
            "TRAIN_SIZE": TRAIN_SIZE,
            "VAL_SIZE": VAL_SIZE,
            "TEST_SIZE": TEST_SIZE,
            "q95_len_train": q95_len_train,
            "HARD_CRITICAL_EVENT_IDS": list(HARD_CRITICAL_EVENT_IDS),
            "SOFT_SUSPICIOUS_EVENT_IDS": list(SOFT_SUSPICIOUS_EVENT_IDS),
        },
        "train_seq_raw": train_seq_raw,
        "val_seq_raw": val_seq_raw,
        "test_seq_raw": test_seq_raw,
        "train_normal_raw": train_normal_raw,
        "train_normal": train_normal,
        "val_seq": val_seq,
        "test_seq": test_seq,
        "annotation_df": annotation_df,
        "template_to_id": template_to_id,
        "id_to_template": id_to_template
    }

    with open(LOCKED_DATA_PATH, "wb") as f:
        pickle.dump(locked_data, f)

    print("\nLocked split saved:", LOCKED_DATA_PATH)

Найден locked split. Загружаю существующее разбиение...
Loaded locked data:
Train raw: (3287, 26)
Train normal raw: (2697, 26)
Train normal chunked: (2697, 31)
Val chunked: (775, 30)
Test chunked: (923, 30)
Annotation: (1698, 33)


In [21]:
def show_split_sizes():
    print("=" * 70)
    print("DATASET SIZES")
    print("=" * 70)

    print("\nRaw windows:")
    print("train_seq_raw:", train_seq_raw.shape)
    print("val_seq_raw:  ", val_seq_raw.shape)
    print("test_seq_raw: ", test_seq_raw.shape)

    print("\nAfter safe-normal filtering and chunking:")
    print("train_normal_raw:", train_normal_raw.shape)
    print("train_normal:    ", train_normal.shape)
    print("val_seq:         ", val_seq.shape)
    print("test_seq:        ", test_seq.shape)

    print("\nAnnotation:")
    print("annotation_df:", annotation_df.shape)

    print("\nSequence length stats:")
    length_stats = pd.DataFrame({
        "train_normal": train_normal["SeqLen"].describe(),
        "val_seq": val_seq["SeqLen"].describe(),
        "test_seq": test_seq["SeqLen"].describe()
    })

    display(length_stats)

    print("\nWeakLabel distribution:")
    for name, df in [
        ("val_seq", val_seq),
        ("test_seq", test_seq),
        ("annotation_df", annotation_df)
    ]:
        if "WeakLabel" in df.columns:
            print(f"\n{name}:")
            print(df["WeakLabel"].value_counts(dropna=False))
            print(df["WeakLabel"].value_counts(normalize=True, dropna=False))


show_split_sizes()

DATASET SIZES

Raw windows:
train_seq_raw: (3287, 26)
val_seq_raw:   (704, 25)
test_seq_raw:  (705, 25)

After safe-normal filtering and chunking:
train_normal_raw: (2697, 26)
train_normal:     (2697, 31)
val_seq:          (775, 30)
test_seq:         (923, 30)

Annotation:
annotation_df: (1698, 33)

Sequence length stats:


,train_normal,val_seq,test_seq
count,2697.000000,775.000000,923.000000
mean,22.654802,27.624516,35.832069
std,9.179782,16.419081,20.204512
min,6.000000,10.000000,10.000000
25%,16.000000,15.000000,17.000000
50%,21.000000,23.000000,30.000000
75%,30.000000,33.000000,64.000000
max,46.000000,64.000000,64.000000



WeakLabel distribution:

val_seq:
WeakLabel
1    467
0    308
Name: count, dtype: int64
WeakLabel
1    0.602581
0    0.397419
Name: proportion, dtype: float64

test_seq:
WeakLabel
1    707
0    216
Name: count, dtype: int64
WeakLabel
1    0.76598
0    0.23402
Name: proportion, dtype: float64

annotation_df:
WeakLabel
1    1174
0     524
Name: count, dtype: int64
WeakLabel
1    0.691402
0    0.308598
Name: proportion, dtype: float64


In [22]:
print("Уникальных последовательностей:")
print("train_normal:", train_normal["Features"].apply(tuple).nunique())
print("val_seq:     ", val_seq["Features"].apply(tuple).nunique())
print("test_seq:    ", test_seq["Features"].apply(tuple).nunique())

print("\nВсего строк:")
print("train_normal:", len(train_normal))
print("val_seq:     ", len(val_seq))
print("test_seq:    ", len(test_seq))

Уникальных последовательностей:
train_normal: 2403
val_seq:      695
test_seq:     870

Всего строк:
train_normal: 2697
val_seq:      775
test_seq:     923


In [23]:
train_set = set(train_normal["Features"].apply(tuple))
val_set = set(val_seq["Features"].apply(tuple))
test_set = set(test_seq["Features"].apply(tuple))

print("Полные повторы последовательностей:")
print("Train ∩ Val :", len(train_set & val_set))
print("Train ∩ Test:", len(train_set & test_set))
print("Val ∩ Test  :", len(val_set & test_set))

print("\nДоли пересечений:")
print("Train ∩ Val / Val unique :", len(train_set & val_set) / max(len(val_set), 1))
print("Train ∩ Test / Test unique:", len(train_set & test_set) / max(len(test_set), 1))
print("Val ∩ Test / Test unique  :", len(val_set & test_set) / max(len(test_set), 1))

Полные повторы последовательностей:
Train ∩ Val : 24
Train ∩ Test: 14
Val ∩ Test  : 18

Доли пересечений:
Train ∩ Val / Val unique : 0.034532374100719423
Train ∩ Test / Test unique: 0.016091954022988506
Val ∩ Test / Test unique  : 0.020689655172413793


## Разметка val/test

In [24]:
def shorten_text(text, max_chars=500):
    text = str(text).replace("\n", " ").replace("\r", " ")
    text = " ".join(text.split())

    if len(text) <= max_chars:
        return text

    return text[:max_chars] + " ...[truncated]"

In [25]:
def build_annotation_prompt(row, max_events=80, max_message_chars=500):
    annotation_id = row["AnnotationID"]
    stable_id = row["StableID"]
    split = row["Split"]

    window_start = row["WindowStart"]
    window_end = row["WindowEnd"]
    seq_len = row["SeqLen"]

    levels = row.get("Levels", [])
    sources = row.get("Sources", [])
    event_ids = row.get("EventIDs", [])
    suspicious_reason = row.get("SuspiciousReason", "")

    features = row.get("Features", [])
    time_intervals = row.get("TimeInterval", [])
    review_templates = row.get("ReviewTemplates", [])
    raw_messages = row.get("RawMessages", [])

    event_lines = []

    n_events = min(len(features), max_events)

    for i in range(n_events):
        feature = features[i] if i < len(features) else ""
        delta_t = time_intervals[i] if i < len(time_intervals) else ""
        template = review_templates[i] if i < len(review_templates) else ""
        message = raw_messages[i] if i < len(raw_messages) else ""

        event_lines.append(
            f"{i + 1}. "
            f"Token: {feature} | "
            f"DeltaTimeSec: {delta_t} | "
            f"Template: {shorten_text(template, max_message_chars)} | "
            f"Message: {shorten_text(message, max_message_chars)}"
        )

    if len(features) > max_events:
        event_lines.append(
            f"... Остальные события не показаны: {len(features) - max_events}"
        )

    events_text = "\n".join(event_lines)

    prompt = f"""
Ты эксперт по анализу Windows Event Logs и обнаружению аномалий в последовательностях событий.

Нужно разметить одну последовательность событий из журналов Windows System/Application.

Классы:

0 = normal
Последовательность похожа на обычную активность системы: информационные события, обычный запуск/остановка служб, Windows Update, BITS, Group Policy, WMI/SCM без признаков сбоя.

1 = anomaly
Последовательность содержит признаки сбоя или нештатного поведения: Error/Critical, Windows Error Reporting, ошибки SQL Server, failed login, unexpected shutdown/restart, service terminated unexpectedly, audit log cleared, repeated crash reports, массовая остановка критичных служб или другая явно подозрительная цепочка.

Важно:
- Не считай Service Control Manager 7036 сам по себе аномалией.
- Не считай обычные циклы WMI Performance Adapter running/stopped аномалией без дополнительных признаков сбоя.
- Не считай Windows Update, BITS, Group Policy или AppX активность аномалией, если нет Error/Critical/WER/SQL-сбоев.
- DCOM timeout или Warning может быть аномалией только в контексте повторяемости, связи с ошибками или другими сбойными событиями.
- Если признаки слабые и нет явных ошибок, выбери label=0, но можешь поставить повышенный anomaly_score.
- Если есть повторяющиеся WER 1001 по sqlservr.exe, MSSQLSERVER Error, failed job SQL Agent, unexpected shutdown/restart или hard critical Event ID, выбери label=1.

Верни ответ строго в JSON формате без дополнительного текста:

{{
  "label": 0 или 1,
  "anomaly_score": число от 0.0 до 1.0,
  "confidence": число от 0.0 до 1.0,
  "key_events": ["список наиболее важных Event ID / источников / токенов"],
  "reason": "краткое объяснение на русском языке"
}}

Метаданные последовательности:
AnnotationID: {annotation_id}
StableID: {stable_id}
Split: {split}
WindowStart: {window_start}
WindowEnd: {window_end}
SeqLen: {seq_len}

Levels в chunk-е:
{levels}

Sources в chunk-е:
{sources}

EventIDs в chunk-е:
{event_ids}

Предварительные weak-label причины chunk-а:
{suspicious_reason}

События последовательности:
{events_text}
""".strip()

    return prompt

In [26]:
annotation_prompts = annotation_df.copy()

annotation_prompts["Prompt"] = annotation_prompts.apply(
    lambda row: build_annotation_prompt(
        row,
        max_events=80,
        max_message_chars=500
    ),
    axis=1
)

prompt_columns = [
    "AnnotationID",
    "StableID",
    "Split",
    "WindowStart",
    "WindowEnd",
    "SeqLen",
    "Levels",
    "Sources",
    "EventIDs",
    "SuspiciousReason",
    "Prompt"
]

annotation_prompts_export = annotation_prompts[prompt_columns].copy()

list_cols = [
    "Levels",
    "Sources",
    "EventIDs"
]

for col in list_cols:
    annotation_prompts_export[col] = annotation_prompts_export[col].apply(
        lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, list) else x
    )

ANNOTATION_PROMPTS_CSV_PATH = f"{PROMPT_EXPORT_DIR}/windows_val_test_annotation_prompts_30min_v3.csv"
ANNOTATION_PROMPTS_JSONL_PATH = f"{PROMPT_EXPORT_DIR}/windows_val_test_annotation_prompts_30min_v3.jsonl"

annotation_prompts_export.to_csv(
    ANNOTATION_PROMPTS_CSV_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("CSV saved:", ANNOTATION_PROMPTS_CSV_PATH)
print("Shape:", annotation_prompts_export.shape)

display(annotation_prompts_export.head())

CSV saved: /content/drive/MyDrive/windows_logs_project/annotation_prompts_30min_v3/windows_val_test_annotation_prompts_30min_v3.csv
Shape: (1698, 11)


,AnnotationID,StableID,Split,WindowStart,WindowEnd,SeqLen,Levels,Sources,EventIDs,SuspiciousReason,Prompt
0,0,val_2026-04-03 16:00:00_2026-04-03 16:30:00_0_...,val,2026-04-03 16:00:00,2026-04-03 16:30:00,17,"[""Information""]","[""MSSQLSERVER"", ""Service Control Manager""]","[""18100"", ""7036""]",,Ты эксперт по анализу Windows Event Logs и обн...
1,1,val_2026-04-03 16:30:00_2026-04-03 17:00:00_0_...,val,2026-04-03 16:30:00,2026-04-03 17:00:00,19,"[""Error"", ""Information""]","[""MSSQLSERVER"", ""Service Control Manager""]","[""1101"", ""7036"", ""7040""]",mssql_error; error_level,Ты эксперт по анализу Windows Event Logs и обн...
2,2,val_2026-04-03 17:00:00_2026-04-03 17:30:00_0_...,val,2026-04-03 17:00:00,2026-04-03 17:30:00,44,"[""Information"", ""Warning""]","[""MSSQLSERVER"", ""Microsoft-Windows-GroupPolicy...","[""1202"", ""1502"", ""158"", ""18100"", ""7036"", ""7040""]",soft_suspicious_event_id:1202,Ты эксперт по анализу Windows Event Logs и обн...
3,3,val_2026-04-03 17:30:00_2026-04-03 18:00:00_0_...,val,2026-04-03 17:30:00,2026-04-03 18:00:00,13,"[""Information""]","[""Service Control Manager""]","[""7036""]",,Ты эксперт по анализу Windows Event Logs и обн...
4,4,val_2026-04-03 18:00:00_2026-04-03 18:30:00_0_...,val,2026-04-03 18:00:00,2026-04-03 18:30:00,24,"[""Information""]","[""MSSQLSERVER"", ""Service Control Manager""]","[""18100"", ""7036""]",,Ты эксперт по анализу Windows Event Logs и обн...


In [27]:
with open(ANNOTATION_PROMPTS_JSONL_PATH, "w", encoding="utf-8") as f:
    for _, row in annotation_prompts.iterrows():
        record = {
            "AnnotationID": int(row["AnnotationID"]),
            "StableID": str(row["StableID"]),
            "Split": str(row["Split"]),
            "WindowStart": str(row["WindowStart"]),
            "WindowEnd": str(row["WindowEnd"]),
            "SeqLen": int(row["SeqLen"]),
            "Prompt": row["Prompt"]
        }

        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("JSONL saved:", ANNOTATION_PROMPTS_JSONL_PATH)

JSONL saved: /content/drive/MyDrive/windows_logs_project/annotation_prompts_30min_v3/windows_val_test_annotation_prompts_30min_v3.jsonl


In [28]:
print(annotation_prompts.loc[0, "Prompt"])

Ты эксперт по анализу Windows Event Logs и обнаружению аномалий в последовательностях событий.

Нужно разметить одну последовательность событий из журналов Windows System/Application.

Классы:

0 = normal
Последовательность похожа на обычную активность системы: информационные события, обычный запуск/остановка служб, Windows Update, BITS, Group Policy, WMI/SCM без признаков сбоя.

1 = anomaly
Последовательность содержит признаки сбоя или нештатного поведения: Error/Critical, Windows Error Reporting, ошибки SQL Server, failed login, unexpected shutdown/restart, service terminated unexpectedly, audit log cleared, repeated crash reports, массовая остановка критичных служб или другая явно подозрительная цепочка.

Важно:
- Не считай Service Control Manager 7036 сам по себе аномалией.
- Не считай обычные циклы WMI Performance Adapter running/stopped аномалией без дополнительных признаков сбоя.
- Не считай Windows Update, BITS, Group Policy или AppX активность аномалией, если нет Error/Critica

In [29]:
ANNOTATION_RESULTS_TEMPLATE_PATH = f"{PROMPT_EXPORT_DIR}/windows_val_test_annotation_results_template_binary_30min_v3.csv"

annotation_results_template = annotation_prompts[[
    "AnnotationID",
    "StableID",
    "Split",
    "WindowStart",
    "WindowEnd",
    "SeqLen"
]].copy()

annotation_results_template["label"] = None
annotation_results_template["anomaly_score"] = None
annotation_results_template["confidence"] = None
annotation_results_template["key_events"] = None
annotation_results_template["reason"] = None
annotation_results_template["annotator"] = None

annotation_results_template.to_csv(
    ANNOTATION_RESULTS_TEMPLATE_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Binary annotation results template saved:", ANNOTATION_RESULTS_TEMPLATE_PATH)

display(annotation_results_template.head())

Binary annotation results template saved: /content/drive/MyDrive/windows_logs_project/annotation_prompts_30min_v3/windows_val_test_annotation_results_template_binary_30min_v3.csv


,AnnotationID,StableID,Split,WindowStart,WindowEnd,SeqLen,label,anomaly_score,confidence,key_events,reason,annotator
0,0,val_2026-04-03 16:00:00_2026-04-03 16:30:00_0_...,val,2026-04-03 16:00:00,2026-04-03 16:30:00,17,None,None,None,None,None,None
1,1,val_2026-04-03 16:30:00_2026-04-03 17:00:00_0_...,val,2026-04-03 16:30:00,2026-04-03 17:00:00,19,None,None,None,None,None,None
2,2,val_2026-04-03 17:00:00_2026-04-03 17:30:00_0_...,val,2026-04-03 17:00:00,2026-04-03 17:30:00,44,None,None,None,None,None,None
3,3,val_2026-04-03 17:30:00_2026-04-03 18:00:00_0_...,val,2026-04-03 17:30:00,2026-04-03 18:00:00,13,None,None,None,None,None,None
4,4,val_2026-04-03 18:00:00_2026-04-03 18:30:00_0_...,val,2026-04-03 18:00:00,2026-04-03 18:30:00,24,None,None,None,None,None,None


In [30]:
THREE_LLM_TEMPLATE_PATH = (
    f"{PROMPT_EXPORT_DIR}/windows_val_test_annotation_results_3llm_30min_v3.csv"
)

annotation_results_3llm = annotation_prompts[[
    "AnnotationID",
    "StableID",
    "Split",
    "WindowStart",
    "WindowEnd",
    "SeqLen"
]].copy()

for model_name in ["llm1", "llm2", "llm3"]:
    annotation_results_3llm[f"{model_name}_label"] = None
    annotation_results_3llm[f"{model_name}_score"] = None
    annotation_results_3llm[f"{model_name}_confidence"] = None
    annotation_results_3llm[f"{model_name}_key_events"] = None
    annotation_results_3llm[f"{model_name}_reason"] = None
    annotation_results_3llm[f"{model_name}_annotator"] = None

annotation_results_3llm.to_csv(
    THREE_LLM_TEMPLATE_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("3-LLM annotation template saved:", THREE_LLM_TEMPLATE_PATH)
display(annotation_results_3llm.head())

3-LLM annotation template saved: /content/drive/MyDrive/windows_logs_project/annotation_prompts_30min_v3/windows_val_test_annotation_results_3llm_30min_v3.csv


,AnnotationID,StableID,Split,WindowStart,WindowEnd,SeqLen,llm1_label,llm1_score,llm1_confidence,llm1_key_events,...,llm2_confidence,llm2_key_events,llm2_reason,llm2_annotator,llm3_label,llm3_score,llm3_confidence,llm3_key_events,llm3_reason,llm3_annotator
0,0,val_2026-04-03 16:00:00_2026-04-03 16:30:00_0_...,val,2026-04-03 16:00:00,2026-04-03 16:30:00,17,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
1,1,val_2026-04-03 16:30:00_2026-04-03 17:00:00_0_...,val,2026-04-03 16:30:00,2026-04-03 17:00:00,19,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2,2,val_2026-04-03 17:00:00_2026-04-03 17:30:00_0_...,val,2026-04-03 17:00:00,2026-04-03 17:30:00,44,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
3,3,val_2026-04-03 17:30:00_2026-04-03 18:00:00_0_...,val,2026-04-03 17:30:00,2026-04-03 18:00:00,13,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
4,4,val_2026-04-03 18:00:00_2026-04-03 18:30:00_0_...,val,2026-04-03 18:00:00,2026-04-03 18:30:00,24,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None


In [31]:
train_normal.to_pickle(f"{EXPORT_DIR}/train_normal.pkl")
val_seq.to_pickle(f"{EXPORT_DIR}/val_seq.pkl")
test_seq.to_pickle(f"{EXPORT_DIR}/test_seq.pkl")

train_seq_raw.to_pickle(f"{EXPORT_DIR}/train_seq_raw.pkl")
val_seq_raw.to_pickle(f"{EXPORT_DIR}/val_seq_raw.pkl")
test_seq_raw.to_pickle(f"{EXPORT_DIR}/test_seq_raw.pkl")

annotation_df.to_pickle(f"{EXPORT_DIR}/annotation_df.pkl")

with open(f"{EXPORT_DIR}/template_to_id.pkl", "wb") as f:
    pickle.dump(template_to_id, f)

with open(f"{EXPORT_DIR}/id_to_template.pkl", "wb") as f:
    pickle.dump(id_to_template, f)

with open(f"{EXPORT_DIR}/template_to_id.json", "w", encoding="utf-8") as f:
    json.dump(template_to_id, f, ensure_ascii=False, indent=2)

with open(f"{EXPORT_DIR}/id_to_template.json", "w", encoding="utf-8") as f:
    json.dump(id_to_template, f, ensure_ascii=False, indent=2)

print("Pickle and dictionary files saved.")

Pickle and dictionary files saved.


In [32]:
def save_df_csv_json_lists(df: pd.DataFrame, path: str):
    df_to_save = df.copy()

    list_cols = [
        "Features",
        "TimeInterval",
        "LogNames",
        "Sources",
        "Levels",
        "EventIDs",
        "RawMessages",
        "ReviewTemplates",
        "EventLogNames",
        "EventSources",
        "EventLevels",
        "EventIDsPerEvent",
        "EventTokens"
    ]

    for col in list_cols:
        if col in df_to_save.columns:
            df_to_save[col] = df_to_save[col].apply(
                lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, list) else x
            )

    df_to_save.to_csv(path, index=False, encoding="utf-8-sig")

In [33]:
save_df_csv_json_lists(train_normal, f"{EXPORT_DIR}/train_normal.csv")
save_df_csv_json_lists(val_seq, f"{EXPORT_DIR}/val_seq.csv")
save_df_csv_json_lists(test_seq, f"{EXPORT_DIR}/test_seq.csv")
save_df_csv_json_lists(annotation_df, f"{EXPORT_DIR}/annotation_df.csv")

print("CSV files saved.")

CSV files saved.


In [34]:
LOCAL_TRAINING_DATA_PATH = f"{EXPORT_DIR}/windows_training_data_30min_v3.pkl"

local_training_data = {
    "params": {
        "VERSION": VERSION,
        "WINDOW_SIZE": WINDOW_SIZE,
        "MIN_SEQ_LEN": MIN_SEQ_LEN,
        "MAX_CHUNK_LEN": MAX_CHUNK_LEN,
        "STRIDE": STRIDE,
        "TRAIN_SIZE": TRAIN_SIZE,
        "VAL_SIZE": VAL_SIZE,
        "TEST_SIZE": TEST_SIZE,
    },
    "train_normal": train_normal,
    "val_seq": val_seq,
    "test_seq": test_seq,
    "annotation_df": annotation_df,
    "template_to_id": template_to_id,
    "id_to_template": id_to_template
}

with open(LOCAL_TRAINING_DATA_PATH, "wb") as f:
    pickle.dump(local_training_data, f)

print("Local training data saved:", LOCAL_TRAINING_DATA_PATH)

Local training data saved: /content/drive/MyDrive/windows_logs_project/export_30min_v3/windows_training_data_30min_v3.pkl


In [35]:
with open(LOCAL_TRAINING_DATA_PATH, "rb") as f:
    check_data = pickle.load(f)

print(check_data.keys())
print("train_normal:", check_data["train_normal"].shape)
print("val_seq:", check_data["val_seq"].shape)
print("test_seq:", check_data["test_seq"].shape)
print("annotation_df:", check_data["annotation_df"].shape)

dict_keys(['params', 'train_normal', 'val_seq', 'test_seq', 'annotation_df', 'template_to_id', 'id_to_template'])
train_normal: (2697, 31)
val_seq: (775, 30)
test_seq: (923, 30)
annotation_df: (1698, 33)
